# humor-me decisions: the model track (judge, SFT, DPO)

Evidence for the trained joke model: the judge that labels preference pairs, then SFT and DPO. Created with its first experiment (Exp 36). Experiment numbers are global across the `decisions*.ipynb` notebooks; the index is [decisions.ipynb](decisions.ipynb).

**How to use it**
- Results load from `reports/metrics/*.json` (cached by parameters), so nothing recomputes unless you ask.
- Every experiment is a `run(**params)` called through `sim.cache.run_or_load`. Set `RERUN = True` after the import in the setup cell to recompute everything.
- Every claim has a **✅ / ❌ check** under it. If a change flips one to ❌, the decision it backs needs revisiting.
- Before committing: `python scripts/run_notebook.py` (restart + run all, clean metadata).

Rules: logic lives in modules; cells only call, display and check. Vocabulary: [glossary](../docs/wiki/glossary.md).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"))
from decisions_common import RERUN, check, show

## Exp 36 Does the judge agree with you on same-writer pairs?

**Decision (D-121, carrying out D-102):** DPO needs thousands of "which bit is funnier" labels, and only a judge can give that many. Before it labels anything it must agree with you on two bits of one request written by the same model (closer than Exp 24's cross-model ballot). You rank 120 pairs: 60 teach the judge your taste, 60 are held back as the test.
**Where it fits:** the model track, between the live writer's bits and DPO training. The pairs come from the live writer (`python -m sim.dpo_pairs write`), half of them with tonight's context.
**Question:** on the 60 test pairs, does the judge pick your bit, in both orders of showing?
**Method:** `python -m sim.dpo_pairs pool`, `write --split ranking`, `rank` (in sittings), `judge`, `score --record`. The judge is Claude Haiku 5.5, thinking off, taught with the 60 teach pairs as examples.
**Bar (D-102, set before any data):** at least 38 of 60 test pairs agree (where the Wilson 95% lower end first clears 50%). A pair where the two orders disagree counts as a miss. Reported, no bar: agreement per tonight half, the share of pairs whose orders disagree, agreement in a single order, ties and skips.
**What is made up:** nothing from `audience.yaml`. Pool sizes (240 and 800), the strict two-order rule and the choice of Haiku are judgment calls, not measured. One day of headlines for the tonight half.
**Outcome:** pending. You run Exp 36 on this branch; the table below fills in then, and a `pass` lets judge labelling of the training split run next.
**How to read:** the checks first (they hold before any answer), then the table. The outcome is shown, not claimed.

In [2]:
from sim import dpo_pairs as dp
from sim import judge_calibration as jc

# Checks that hold before any answer: the bar and the design are the ones decided.
check("The bar is D-102's: at least 38 of 60 test pairs, and 38 is exactly where the Wilson lower end first clears 50%",
      (dp.BAR_AGREE, dp.N_TEST) == (38, 60) and jc.wilson(38, 60)[0] > 0.5 >= jc.wilson(37, 60)[0])
check("120 decided pairs are 4 cells of 30: teach and test, tonight half and plain half",
      dp.N_PAIRS == 120 and dp.CELLS == (("teach", "tonight"), ("test", "tonight"), ("teach", "plain"), ("test", "plain")) and dp.PER_CELL == 30)

# The pool on a made-up day (no stored headline text): deterministic, split disjoint, every cell big enough.
day = {"date": "2000-01-01", "approved": [f"gist {i}" for i in range(5)], "facts": [], "market": "up", "names": [], "people": []}
pool = dp.build_pool([day])
check("The pool is deterministic for a seed and its sizes are the decided ones",
      dp.build_pool([day]) == pool and dp.sizes(pool) == {"ranking/tonight": 120, "ranking/plain": 120, "train/tonight": 400, "train/plain": 400})
dp.assert_disjoint(pool)  # raises if a ranking or training prompt repeats Exp 24's ballot or Exp 26, 27, 30, or the splits overlap
check("No ranking or training prompt repeats one of Exp 24's ballot or Exp 26, 27 and 30, and the two splits share none (assert_disjoint passed)", True)
check("Roles are fixed in the pool before any answer: 60 prompts of each role in each tonight half of the ranking split",
      all(sum(p["split"] == "ranking" and p["half"] == h and p["role"] == r for p in pool["prompts"]) == 60 for h in dp.HALVES for r in dp.ROLES))
check("Headline topics sit on bits 2 and 6 only, and only in the tonight half",
      {p["bit"] for p in pool["prompts"] if p["gist"]} == {2, 6} and all(p["half"] == "tonight" for p in pool["prompts"] if p["gist"]))

res = dp.stored()
if res is None:
    show("**Pending.** Rank 120 pairs (`python -m sim.dpo_pairs rank`), then `judge` and `score --record` (Ollama up for `write`, `ANTHROPIC_API_KEY` set for `judge`), then re-run this notebook.")
else:
    show(dp.table(res))
    show(f"**Script verdict: `{res['verdict']}`.** Judge: {res['judge']['model']}, taught with {res['judge']['teach_pairs']} pairs.")
    rule = "pass" if res["agree"] >= dp.BAR_AGREE and res["wilson95"][0] > 0.5 else "fail"
    check("The recorded verdict is what the fixed rule gives for the recorded numbers",
          res["verdict"] == (rule if res["n_test"] == dp.N_TEST and not res["missing_pairs"] else "incomplete"))
    check("The two halves add up to the test pairs", sum(h["pairs"] for h in res["by_half"].values()) == res["n_test"])


✅ **The bar is D-102's: at least 38 of 60 test pairs, and 38 is exactly where the Wilson lower end first clears 50%**

✅ **120 decided pairs are 4 cells of 30: teach and test, tonight half and plain half**

✅ **The pool is deterministic for a seed and its sizes are the decided ones**

✅ **No ranking or training prompt repeats one of Exp 24's ballot or Exp 26, 27 and 30, and the two splits share none (assert_disjoint passed)**

✅ **Roles are fixed in the pool before any answer: 60 prompts of each role in each tonight half of the ranking split**

✅ **Headline topics sit on bits 2 and 6 only, and only in the tonight half**

**Pending.** Rank 120 pairs (`python -m sim.dpo_pairs rank`), then `judge` and `score --record` (Ollama up for `write`, `ANTHROPIC_API_KEY` set for `judge`), then re-run this notebook.